In [ ]:
!pip install gradio timm opencv-python-headless pillow reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import cv2
from PIL import Image
from datetime import datetime
import tempfile
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import gradio as gr

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch, mm
from reportlab.lib.colors import HexColor, black, white
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak, HRFlowable,
)
from reportlab.pdfgen import canvas as pdf_canvas

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR = '/content/drive/MyDrive/models'
REPORT_DIR = '/content/drive/MyDrive/DermaScan_Reports'

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

EFF_WEIGHT  = 0.45
CONV_WEIGHT = 0.55

# Create report directory on Drive
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
class SpatialAttentionGate(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels // 4, 1)
        self.conv2 = nn.Conv2d(in_channels // 4, 1, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        attn = F.gelu(self.conv1(x))
        attn = self.sigmoid(self.conv2(attn))
        return x * attn, attn


class LesionFocusedEfficientNet(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnetv2_m.in21k_ft_in1k',
            pretrained=pretrained,
            features_only=False,
            num_classes=0,
            global_pool=''
        )
        feat_dim = self.backbone.num_features
        self.spatial_attn = SpatialAttentionGate(feat_dim)
        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward_features(self, x):
        return self.backbone.forward_features(x)

    def forward(self, x, return_attention=False):
        feat_map = self.forward_features(x)
        attended, attn_map = self.spatial_attn(feat_map)
        pooled = attended.mean(dim=[2, 3])
        pooled = self.norm(pooled)
        logits = self.head(pooled)
        if return_attention:
            return logits, attn_map, feat_map
        return logits


class LesionFocusedConvNeXt(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_base.fb_in1k',
            pretrained=pretrained,
            features_only=False,
            num_classes=0
        )
        feat_dim = self.backbone.num_features
        self.spatial_attn = SpatialAttentionGate(feat_dim)
        self.backbone.head.global_pool = nn.Identity()
        self.backbone.head.norm = nn.Identity()
        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward_features(self, x):
        return self.backbone.forward_features(x)

    def forward(self, x, return_attention=False):
        feat_map = self.forward_features(x)
        attended, attn_map = self.spatial_attn(feat_map)
        pooled = attended.mean(dim=[2, 3])
        pooled = self.norm(pooled)
        logits = self.head(pooled)
        if return_attention:
            return logits, attn_map, feat_map
        return logits



In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model

    def generate(self, input_tensor):
        self.model.eval()
        input_tensor = input_tensor.clone().detach().requires_grad_(True)

        feat_map = self.model.forward_features(input_tensor)
        feat_map.retain_grad()

        attended, _ = self.model.spatial_attn(feat_map)
        pooled = attended.mean(dim=[2, 3])
        pooled = self.model.norm(pooled)
        logits = self.model.head(pooled)
        prob = torch.sigmoid(logits)

        self.model.zero_grad()
        logits.backward(retain_graph=True)

        gradients = feat_map.grad
        if gradients is None:
            return prob.item(), None

        weights = gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * feat_map).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().detach().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()

        return prob.item(), cam

In [ ]:
def preprocess_image(image_np, img_size):
    img = cv2.resize(image_np, (img_size, img_size))
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    img = img.transpose(2, 0, 1)
    tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
    return tensor.to(DEVICE)


def cam_to_heatmap(cam, original_img):
    if cam is None:
        h, w = original_img.shape[:2]
        return np.zeros((h, w, 3), dtype=np.uint8)
    h, w = original_img.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    cam_uint8 = np.uint8(255 * cam_resized)
    heatmap = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = 0.5 * original_img.astype(np.float32) + 0.5 * heatmap.astype(np.float32)
    return np.clip(overlay, 0, 255).astype(np.uint8)


def add_label(img, text, color=(255, 255, 255)):
    """Add a text label at the top of an image."""
    img = img.copy()
    cv2.rectangle(img, (0, 0), (img.shape[1], 36), (0, 0, 0), -1)
    cv2.putText(img, text, (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img



In [ ]:
from reportlab.lib.colors import HexColor
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak, HRFlowable,
)

class DermaScanReport:
    """Generates a professional PDF report for DermaScan predictions."""

    # Color palette
    PRIMARY    = HexColor('#6C3FA0')   # Purple
    SECONDARY  = HexColor('#8B5CF6')   # Lighter purple
    DARK_BG    = HexColor('#1E1B2E')   # Dark background
    LIGHT_BG   = HexColor('#F5F3FF')   # Light purple bg
    TEXT_DARK  = HexColor('#1F2937')
    TEXT_GRAY  = HexColor('#6B7280')
    RED        = HexColor('#DC2626')
    YELLOW     = HexColor('#F59E0B')
    GREEN      = HexColor('#16A34A')

    def __init__(self):
        self.styles = getSampleStyleSheet()
        self._setup_styles()

    def _setup_styles(self):
        """Create custom paragraph styles for the report."""
        self.styles.add(ParagraphStyle(
            'ReportTitle',
            parent=self.styles['Title'],
            fontSize=26,
            textColor=self.PRIMARY,
            spaceAfter=6,
            alignment=TA_CENTER,
            fontName='Helvetica-Bold',
        ))
        self.styles.add(ParagraphStyle(
            'ReportSubtitle',
            parent=self.styles['Normal'],
            fontSize=11,
            textColor=self.TEXT_GRAY,
            alignment=TA_CENTER,
            spaceAfter=20,
        ))
        self.styles.add(ParagraphStyle(
            'SectionHead',
            parent=self.styles['Heading2'],
            fontSize=14,
            textColor=self.PRIMARY,
            spaceBefore=18,
            spaceAfter=8,
            fontName='Helvetica-Bold',
        ))
        self.styles.add(ParagraphStyle(
            'BodyText2',
            parent=self.styles['Normal'],
            fontSize=10,
            textColor=self.TEXT_DARK,
            spaceAfter=6,
            leading=14,
        ))
        self.styles.add(ParagraphStyle(
            'Disclaimer',
            parent=self.styles['Normal'],
            fontSize=8,
            textColor=self.TEXT_GRAY,
            alignment=TA_CENTER,
            spaceBefore=12,
            leading=10,
        ))

    def _header_footer(self, canvas, doc):
        """Draw header bar and footer on each page."""
        canvas.saveState()
        w, h = A4

        # Header bar
        canvas.setFillColor(self.PRIMARY)
        canvas.rect(0, h - 40, w, 40, fill=True, stroke=False)
        canvas.setFillColor(white)
        canvas.setFont('Helvetica-Bold', 12)
        canvas.drawString(20, h - 28, "DermaScan Ensemble — Melanoma Detection Report")
        canvas.setFont('Helvetica', 8)
        canvas.drawRightString(w - 20, h - 28, datetime.now().strftime('%B %d, %Y  %I:%M %p'))

        # Footer
        canvas.setFillColor(self.TEXT_GRAY)
        canvas.setFont('Helvetica', 7)
        canvas.drawString(20, 20, "DermaScan Ensemble | EfficientNetV2-M + ConvNeXt-Base")
        canvas.drawRightString(w - 20, 20, f"Page {doc.page}")

        canvas.restoreState()

    def generate(self, patient_name, prediction, risk_level,
                 eff_prob, conv_prob, ensemble_prob,
                 original_img_np, eff_overlay_np, conv_overlay_np,
                 ensemble_overlay_np):
        """
        Generate the PDF report and return (local_path, drive_path).
        All image inputs are numpy RGB arrays.
        """
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        safe_name = patient_name.replace(' ', '_').replace('/', '_')
        filename = f"DermaScan_Report_{safe_name}_{timestamp}.pdf"

        # Work in a temp directory then copy to Drive
        tmp_dir = tempfile.mkdtemp()
        local_pdf_path = os.path.join(tmp_dir, filename)

        # Save images as temp files for ReportLab
        img_paths = {}
        for name, arr in [('original', original_img_np),
                          ('efficientnet', eff_overlay_np),
                          ('convnext', conv_overlay_np),
                          ('ensemble', ensemble_overlay_np)]:
            path = os.path.join(tmp_dir, f'{name}.png')
            Image.fromarray(arr).save(path)
            img_paths[name] = path

        # Build PDF
        doc = SimpleDocTemplate(
            local_pdf_path,
            pagesize=A4,
            topMargin=55,
            bottomMargin=40,
            leftMargin=40,
            rightMargin=40,
        )

        story = []

        # ── Title section ──
        story.append(Spacer(1, 10))
        story.append(Paragraph("DermaScan Analysis Report", self.styles['ReportTitle']))
        story.append(Paragraph(
            "EfficientNetV2-M + ConvNeXt-Base Ensemble with Grad-CAM",
            self.styles['ReportSubtitle']
        ))

        # ── Patient info table ──
        story.append(Paragraph("Patient Information", self.styles['SectionHead']))

        risk_color = {
            'HIGH': self.RED, 'MODERATE': self.YELLOW, 'LOW': self.GREEN
        }.get(risk_level, self.TEXT_DARK)

        info_data = [
            ['Patient Name', patient_name],
            ['Date & Time', datetime.now().strftime('%B %d, %Y at %I:%M %p')],
            ['Device', str(DEVICE).upper()],
        ]
        info_table = Table(info_data, colWidths=[150, 340])
        info_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (0, -1), self.LIGHT_BG),
            ('TEXTCOLOR', (0, 0), (0, -1), self.PRIMARY),
            ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
            ('FONTNAME', (1, 0), (1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('TOPPADDING', (0, 0), (-1, -1), 6),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
            ('LEFTPADDING', (0, 0), (-1, -1), 10),
            ('GRID', (0, 0), (-1, -1), 0.5, HexColor('#E5E7EB')),
            ('ROUNDEDCORNERS', [4, 4, 4, 4]),
        ]))
        story.append(info_table)

        # ── Prediction results ──
        story.append(Spacer(1, 6))
        story.append(Paragraph("Prediction Results", self.styles['SectionHead']))

        risk_label = f'<font color="#{risk_color.hexval()[2:]}">{risk_level} RISK</font>'

        results_data = [
            ['Model', 'Probability'],
            ['EfficientNetV2-M (480x480)', f'{eff_prob * 100:.1f}%'],
            ['ConvNeXt-Base (288x288)', f'{conv_prob * 100:.1f}%'],
            ['Ensemble (50/50)', f'{ensemble_prob * 100:.1f}%'],
        ]
        results_table = Table(results_data, colWidths=[300, 190])
        results_table.setStyle(TableStyle([
            # Header row
            ('BACKGROUND', (0, 0), (-1, 0), self.PRIMARY),
            ('TEXTCOLOR', (0, 0), (-1, 0), white),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 10),
            # Ensemble row highlight
            ('BACKGROUND', (0, 3), (-1, 3), self.LIGHT_BG),
            ('FONTNAME', (0, 3), (-1, 3), 'Helvetica-Bold'),
            # General
            ('FONTNAME', (0, 1), (-1, 2), 'Helvetica'),
            ('FONTSIZE', (0, 1), (-1, -1), 10),
            ('ALIGN', (1, 0), (1, -1), 'CENTER'),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('TOPPADDING', (0, 0), (-1, -1), 7),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 7),
            ('LEFTPADDING', (0, 0), (-1, -1), 10),
            ('GRID', (0, 0), (-1, -1), 0.5, HexColor('#E5E7EB')),
        ]))
        story.append(results_table)

        # ── Verdict box ──
        story.append(Spacer(1, 10))
        verdict_text = f'Classification: <b>{prediction}</b>  |  Risk Level: <b>{risk_level}</b>  |  Ensemble Score: <b>{ensemble_prob * 100:.1f}%</b>'
        verdict_data = [[Paragraph(verdict_text, ParagraphStyle(
            'Verdict', parent=self.styles['Normal'],
            fontSize=11, textColor=self.TEXT_DARK, alignment=TA_CENTER,
        ))]]
        verdict_box = Table(verdict_data, colWidths=[490])
        box_bg = HexColor('#FEF2F2') if prediction == 'Melanoma' else HexColor('#F0FDF4')
        border_c = self.RED if prediction == 'Melanoma' else self.GREEN
        verdict_box.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, -1), box_bg),
            ('BOX', (0, 0), (-1, -1), 1.5, border_c),
            ('TOPPADDING', (0, 0), (-1, -1), 10),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 10),
        ]))
        story.append(verdict_box)

        # ── Original image ──
        story.append(Spacer(1, 10))
        story.append(Paragraph("Uploaded Image", self.styles['SectionHead']))
        story.append(RLImage(img_paths['original'], width=220, height=220))

        # ── Grad-CAM heatmaps ──
        story.append(Spacer(1, 8))
        story.append(Paragraph("Grad-CAM Heatmaps", self.styles['SectionHead']))
        story.append(Paragraph(
            "Heatmaps show which regions of the image each model focused on. "
            "Warm colors (red/yellow) indicate high attention areas.",
            self.styles['BodyText2']
        ))

        cam_imgs = [
            [Paragraph('<b>EfficientNetV2-M</b>', ParagraphStyle('c', alignment=TA_CENTER, fontSize=9)),
             Paragraph('<b>ConvNeXt-Base</b>', ParagraphStyle('c', alignment=TA_CENTER, fontSize=9)),
             Paragraph('<b>Ensemble</b>', ParagraphStyle('c', alignment=TA_CENTER, fontSize=9))],
            [RLImage(img_paths['efficientnet'], width=155, height=155),
             RLImage(img_paths['convnext'], width=155, height=155),
             RLImage(img_paths['ensemble'], width=155, height=155)],
        ]
        cam_table = Table(cam_imgs, colWidths=[165, 165, 165])
        cam_table.setStyle(TableStyle([
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('TOPPADDING', (0, 0), (-1, -1), 4),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
        ]))
        story.append(cam_table)

        # ── Disclaimer ──
        story.append(Spacer(1, 20))
        story.append(HRFlowable(width="100%", thickness=0.5, color=HexColor('#E5E7EB')))
        story.append(Paragraph(
            "<b>DISCLAIMER:</b> This report is generated by an AI-based research tool for "
            "<b>educational and research purposes only</b>. It is NOT a medical diagnosis. "
            "The predictions should not be used as a substitute for professional medical advice. "
            "Always consult a qualified dermatologist for skin lesion evaluation.",
            self.styles['Disclaimer']
        ))

        # Build
        doc.build(story, onFirstPage=self._header_footer, onLaterPages=self._header_footer)

        # Copy to Google Drive
        drive_path = os.path.join(REPORT_DIR, filename)
        shutil.copy2(local_pdf_path, drive_path)

        return local_pdf_path, drive_path

In [ ]:
print("=" * 50)
print("Loading models from Google Drive...")
print("=" * 50)

print("  Loading EfficientNetV2-M...")
eff_model = LesionFocusedEfficientNet(pretrained=False).to(DEVICE)
eff_path = os.path.join(MODEL_DIR, 'melanomaEfficientNetV2M_lesion1.pth')
eff_model.load_state_dict(torch.load(eff_path, map_location=DEVICE, weights_only=True))
eff_model.eval()
print("  ✓ EfficientNetV2-M loaded")

print("  Loading ConvNeXt-Base...")
conv_model = LesionFocusedConvNeXt(pretrained=False).to(DEVICE)
conv_path = os.path.join(MODEL_DIR, 'melanomaConvNeXt_lesion1.pth')
conv_model.load_state_dict(torch.load(conv_path, map_location=DEVICE, weights_only=True))
conv_model.eval()
print("  ✓ ConvNeXt-Base loaded")

eff_gradcam  = GradCAM(eff_model)
conv_gradcam = GradCAM(conv_model)

report_gen = DermaScanReport()

print(f"\n✅ Both models ready on {DEVICE}")
print(f"📁 Reports will be saved to: {REPORT_DIR}")
print("=" * 50)


Loading models from Google Drive...
  Loading EfficientNetV2-M...
  ✓ EfficientNetV2-M loaded
  Loading ConvNeXt-Base...
  ✓ ConvNeXt-Base loaded

✅ Both models ready on cuda
📁 Reports will be saved to: /content/drive/MyDrive/DermaScan_Reports


In [ ]:
latest_prediction = {}

In [ ]:
def predict(input_image):
    """
    Main prediction function called by Gradio.
    Takes a PIL Image, returns (result_text, gradcam_gallery).
    """
    global latest_prediction

    if input_image is None:
        return "⚠️ Please upload an image.", [], gr.update(interactive=False)

    # Convert to numpy RGB
    img_np = np.array(input_image.convert('RGB'))

    # ── Run both models ──
    eff_input = preprocess_image(img_np, 480)
    eff_prob, eff_cam = eff_gradcam.generate(eff_input)

    conv_input = preprocess_image(img_np, 288)
    conv_prob, conv_cam = conv_gradcam.generate(conv_input)

    # ── Ensemble ──
    ensemble_prob = EFF_WEIGHT * eff_prob + CONV_WEIGHT * conv_prob

    # ── Classification ──
    prediction = 'Melanoma' if ensemble_prob >= 0.5 else 'Benign'
    if ensemble_prob >= 0.7:
        risk_level, risk_emoji = 'HIGH', '🔴'
    elif ensemble_prob >= 0.4:
        risk_level, risk_emoji = 'MODERATE', '🟡'
    else:
        risk_level, risk_emoji = 'LOW', '🟢'

    # ── Build result text ──
    result_text = f"""
## {risk_emoji} {prediction} — {risk_level} Risk

| Model | Probability |
|-------|------------|
| EfficientNetV2-M (480×480) | **{eff_prob*100:.1f}%** |
| ConvNeXt-Base (288×288) | **{conv_prob*100:.1f}%** |
| **Ensemble (50/50)** | **{ensemble_prob*100:.1f}%** |

---
*Device: {DEVICE} • Threshold: 0.5*
"""

    # ── Generate Grad-CAM gallery ──
    display_img = cv2.resize(img_np, (480, 480))

    original_labeled  = add_label(display_img, "Original")
    eff_overlay       = add_label(cam_to_heatmap(eff_cam, display_img), f"EfficientNetV2-M  ({eff_prob*100:.1f}%)")
    conv_overlay      = add_label(cam_to_heatmap(conv_cam, display_img), f"ConvNeXt-Base  ({conv_prob*100:.1f}%)")

    # Combined Grad-CAM
    if eff_cam is not None and conv_cam is not None:
        eff_r  = cv2.resize(eff_cam,  (480, 480))
        conv_r = cv2.resize(conv_cam, (480, 480))
        combined = 0.5 * eff_r + 0.5 * conv_r
        if combined.max() > 0:
            combined = combined / combined.max()
        ensemble_overlay = add_label(cam_to_heatmap(combined, display_img), f"Ensemble  ({ensemble_prob*100:.1f}%)")
    else:
        ensemble_overlay = add_label(display_img, "Ensemble (no CAM)")

    gallery = [
        (original_labeled,  "Original"),
        (eff_overlay,       "EfficientNetV2-M"),
        (conv_overlay,      "ConvNeXt-Base"),
        (ensemble_overlay,  "Ensemble"),
    ]

    # ── Store results for PDF generation ──
    latest_prediction = {
        'prediction': prediction,
        'risk_level': risk_level,
        'eff_prob': eff_prob,
        'conv_prob': conv_prob,
        'ensemble_prob': ensemble_prob,
        'original_img': display_img.copy(),
        'eff_overlay': eff_overlay.copy(),
        'conv_overlay': conv_overlay.copy(),
        'ensemble_overlay': ensemble_overlay.copy(),
    }

    return result_text, gallery, gr.update(interactive=True)



In [ ]:
def generate_report(patient_name):
    """Generate PDF report and return file path for download."""
    global latest_prediction

    if not latest_prediction:
        return None, "⚠️ Please analyze an image first before generating a report."

    if not patient_name or patient_name.strip() == '':
        return None, "⚠️ Please enter your name to generate the report."

    patient_name = patient_name.strip()

    try:
        local_path, drive_path = report_gen.generate(
            patient_name=patient_name,
            prediction=latest_prediction['prediction'],
            risk_level=latest_prediction['risk_level'],
            eff_prob=latest_prediction['eff_prob'],
            conv_prob=latest_prediction['conv_prob'],
            ensemble_prob=latest_prediction['ensemble_prob'],
            original_img_np=latest_prediction['original_img'],
            eff_overlay_np=latest_prediction['eff_overlay'],
            conv_overlay_np=latest_prediction['conv_overlay'],
            ensemble_overlay_np=latest_prediction['ensemble_overlay'],
        )

        status_msg = (
            f"✅ Report generated successfully!\n\n"
            f"📄 **Patient:** {patient_name}\n"
            f"💾 **Saved to Google Drive:**\n`{drive_path}`\n\n"
            f"⬇️ Click the download link below to save locally."
        )

        return local_path, status_msg

    except Exception as e:
        return None, f"❌ Error generating report: {str(e)}"

In [ ]:
css = """
.gradio-container { max-width: 950px !important; }
.result-box { font-size: 16px; }
.report-section {
    border: 1px solid #8B5CF6;
    border-radius: 8px;
    padding: 12px;
    margin-top: 12px;
    background: linear-gradient(135deg, #F5F3FF 0%, #EDE9FE 100%);
}
"""

with gr.Blocks(
    title="DermaScan Ensemble",
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="purple",
        neutral_hue="slate",
    ),
    css=css,
) as demo:

    gr.Markdown("""
    # 🔬 DermaScan Ensemble — Melanoma Detection
    **EfficientNetV2-M + ConvNeXt-Base** with Grad-CAM visualization

    Upload a dermoscopy image to get a melanoma probability prediction
    with interpretability heatmaps from both models.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(
                type="pil",
                label="Upload Dermoscopy Image",
                height=400,
            )
            analyze_btn = gr.Button(
                "🔍 Analyze with Ensemble",
                variant="primary",
                size="lg",
            )

        with gr.Column(scale=1):
            result_md = gr.Markdown(
                label="Results",
                elem_classes=["result-box"],
            )

    gr.Markdown("### Grad-CAM Heatmaps")
    gradcam_gallery = gr.Gallery(
        label="Grad-CAM Visualization",
        columns=4,
        rows=1,
        height=350,
        object_fit="contain",
    )

    # ── PDF Report Section ──
    gr.Markdown("---")
    gr.Markdown("### 📄 Generate PDF Report")

    with gr.Row(elem_classes=["report-section"]):
        with gr.Column(scale=2):
            patient_name_input = gr.Textbox(
                label="Patient / Your Name",
                placeholder="Enter name for the report...",
                max_lines=1,
            )
            generate_btn = gr.Button(
                "📄 Generate PDF Report",
                variant="secondary",
                size="lg",
                interactive=False,
            )
        with gr.Column(scale=2):
            report_status = gr.Markdown(
                value="*Analyze an image first, then enter your name to generate a report.*"
            )
            report_file = gr.File(
                label="Download Report",
                visible=True,
            )

    gr.Markdown("""
    ---
    > ⚠️ **Disclaimer:** This tool is for **research and educational purposes only**.
    > It is not a medical device and should not be used for clinical diagnosis.
    > Always consult a qualified dermatologist for skin lesion evaluation.
    """)

    # ── Wire up ──
    analyze_btn.click(
        fn=predict,
        inputs=[input_image],
        outputs=[result_md, gradcam_gallery, generate_btn],
    )

    input_image.change(
        fn=predict,
        inputs=[input_image],
        outputs=[result_md, gradcam_gallery, generate_btn],
    )

    generate_btn.click(
        fn=generate_report,
        inputs=[patient_name_input],
        outputs=[report_file, report_status],
    )


/tmp/ipykernel_2960/2343707338.py:13: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2960/2343707338.py:13: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


In [ ]:
demo.launch(
    share=True,       # Creates a public gradio.live link
    debug=True,
    show_error=True,
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://22621743163e70532d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
